In [ ]:
pip install Flask Flask-MySQLdb Werkzeug
CREATE TABLE guru (
    id INT AUTO_INCREMENT PRIMARY KEY,
    username VARCHAR(50) NOT NULL UNIQUE,
    password VARCHAR(255) NOT NULL,
    nama_guru VARCHAR(100) NOT NULL
);

-- Contoh data guru (password asli: "rahasia123", sudah di-hash agar aman)
INSERT INTO guru (username, password, nama_guru) VALUES 
('budi_guru', 'scrypt:32768:8:1$vB87XG9wO8M5zZ1B$e6bfbd3b0a02f37c...', 'Budi Santoso, S.Pd.');


In [ ]:
from flask import Flask, render_template, request, redirect, url_for, session, flash
from flask_mysqldb import MySQL
from werkzeug.security import check_password_hash, generate_password_hash

app = Flask(__name__)

# Konfigurasi Keamanan & Database
app.secret_key = 'kunci_rahasia_anda_yang_sangat_kuat'
app.config['MYSQL_HOST'] = 'localhost'
app.config['MYSQL_USER'] = 'root'
app.config['MYSQL_PASSWORD'] = ''
app.config['MYSQL_DB'] = 'nama_database_anda'

mysql = MySQL(app)

# 1. RUTE LOGIN GURU
@app.route('/login', methods=['GET', 'POST'])
def login():
    # Jika sudah login, langsung alihkan ke halaman input nilai
    if 'guru_logged_in' in session:
        return redirect(url_for('input_nilai'))

    if request.method == 'POST':
        username = request.form['username'].strip()
        password = request.form['password'].strip()

        cursor = mysql.connection.cursor()
        cursor.execute("SELECT id, username, password, nama_guru FROM guru WHERE username = %s", (username,))
        guru = cursor.fetchone()
        cursor.close()

        if guru:
            # guru[2] adalah kolom password yang di-hash di database
            if check_password_hash(guru[2], password):
                # Simpan data ke session
                session['guru_logged_in'] = True
                session['guru_id'] = guru[0]
                session['guru_nama'] = guru[3]
                return redirect(url_for('input_nilai'))
            else:
                flash('Password salah!', 'danger')
        else:
            flash('Username tidak ditemukan!', 'danger')

    return render_template('login.html')

# 2. RUTE HALAMAN UTAMA (PROTEKSI INPUT NILAI)
@app.route('/input-nilai')
def input_nilai():
    # Proteksi: Jika belum login, tendang balik ke halaman login
    if 'guru_logged_in' not in session:
        flash('Silakan login terlebih dahulu!', 'warning')
        return redirect(url_for('login'))
    
    return render_template('input_nilai.html', nama=session['guru_nama'])

# 3. RUTE LOGOUT
@app.route('/logout')
def logout():
    session.clear()
    flash('Anda telah berhasil keluar.', 'success')
    return redirect(url_for('login'))

if __name__ == '__main__':
    app.run(debug=True)
